# 大型遥感影像分块处理 (Part 1 - 64x64 版本)

**目标**: 将一张大型卫星图像切割成一系列 **64x64** 像素的小图像块（Patches），并将其保存到磁盘，为后续的模型推理做准备。

**背景**: 根据你的描述，你所训练的ViT模型接受的是 64x64 像素的图像。因此，我们采用**滑动窗口（Sliding Window）**的方法，将大图分解为模型可以处理的 64x64 小块。

为了减少图像块边缘的分类错误效应，我们使用带有**重叠（Overlap）**的窗口。这通过设置一个小于图像块尺寸的**步长（Stride）**来实现。

### 步骤 1: 导入必要的库

In [ ]:
import os
from PIL import Image
from tqdm import tqdm  # 用于显示进度条，方便监控进度
import matplotlib.pyplot as plt

# 增加 PIL 对大图的支持，以防在处理高分辨率影像时出现 DecompressionBombError
Image.MAX_IMAGE_PIXELS = None

### 步骤 2: 定义参数和文件路径

**关键配置区域**：在这里，你需要设置你的大图路径、输出目录，以及最重要的——将切割尺寸设为 `64`。

In [ ]:
# --- 用户配置区域 ---

# 1. 输入的大型卫星图像路径
# 请将这里替换为你的2500x1131图像的实际路径
large_image_path = "large_satellite_image.jpg"  

# 2. 切割后的小图像块保存目录
output_dir = "./patches_64x64/"  # 建议使用一个新目录名以区分不同尺寸

# 3. 图像块的尺寸 (必须与你ViT模型训练时的输入尺寸完全一致)
patch_size = 64

# 4. 滑动窗口的步长。这里我们设为 32，表示有 50% 的重叠区域。
# 如果不希望有重叠，可以将 stride 设置为 64。
stride = 32 

# --- 配置结束 ---

### 步骤 3: 创建输出目录

In [ ]:
os.makedirs(output_dir, exist_ok=True)
print(f"图像块将被保存在: '{os.path.abspath(output_dir)}' 目录中。")

### 步骤 4: 加载大图并进行分块与保存

此代码块会遍历整个大图，按照 `64x64` 的尺寸和 `32` 的步长进行切割，并将每个小块以 `patch_行号_列号.png` 的格式保存。这个命名规则对于后续拼接至关重要。

In [ ]:
try:
    # 加载大图
    large_image = Image.open(large_image_path).convert("RGB")
    width, height = large_image.size
    print(f"成功加载图像: {large_image_path}")
    print(f"图像尺寸: {width} x {height}")
    
    patch_count = 0
    
    # 使用tqdm来可视化处理进度
    # 保证窗口不会超出图像边界
    for y in tqdm(range(0, height - patch_size + 1, stride), desc="处理行"):
        for x in range(0, width - patch_size + 1, stride):
            # 定义切割区域: (left, upper, right, lower)
            box = (x, y, x + patch_size, y + patch_size)
            patch = large_image.crop(box)
            
            # 计算行列索引，用于文件名
            row = y // stride
            col = x // stride
            
            # 构建文件名并保存 (PNG格式是无损的，推荐使用)
            filename = f"patch_{row}_{col}.png"
            output_path = os.path.join(output_dir, filename)
            patch.save(output_path)
            
            patch_count += 1

    print(f"\n处理完成！总共保存了 {patch_count} 个 64x64 的图像块到 '{output_dir}' 目录。")

except FileNotFoundError:
    print(f"错误：找不到文件 '{large_image_path}'。请检查路径是否正确。")
except Exception as e:
    print(f"处理过程中发生错误: {e}")

### (可选) 步骤 5: 验证并可视化一个样本

加载一个刚刚生成的图像块，确认其尺寸和内容是否正确。

In [ ]:
# 随机选择一个图像块进行显示
sample_patch_path = os.path.join(output_dir, "patch_0_0.png")

if os.path.exists(sample_patch_path):
    print("显示一个样本图像块...")
    sample_image = Image.open(sample_patch_path)
    
    plt.imshow(sample_image)
    plt.title(f"样本图像块 (尺寸: {sample_image.size[0]}x{sample_image.size[1]})")
    plt.axis('off')
    plt.show()
else:
    print("未找到样本图像块，请检查切割过程是否成功。")

### 总结与下一步

至此，我们已经成功地将大型遥感影像转换为了 **64x64** 像素的小图像块，并保存在了 `./patches_64x64/` 文件夹中。

**下一步（Part 2）** 将是创建另一个Notebook，用于加载这些图像块，使用你的ViT模型进行分类，并生成最终的土地覆盖地图。